In [1]:
import os
DOTENV_PATH = '../../apis/.env'
import dotenv
dotenv.load_dotenv(DOTENV_PATH)
hf_token_write = os.getenv('HF_TOKEN_WRITE')
def mask_token(token):
    return token[:4] + '*' * (len(token) - 8) + token[-4:]
print(f"HF_TOKEN_WRITE: {mask_token(hf_token_write)}")

from sentence_transformers import SentenceTransformer

from sentence_transformers.models import StaticEmbedding
from datasets import load_dataset
import duckdb
from typing import List

import time
def play_chimes():
    sound_path = r"C:\Windows\Media\chimes.wav"
    os.system(f'powershell -c (New-Object Media.SoundPlayer "{sound_path}").PlaySync();')
play_chimes() # Play the sound when this function runs (I use this to signal the end of long tasks)

HF_TOKEN_WRITE: hf_u*****************************Xipx



In [2]:
static_embedding = StaticEmbedding.from_model2vec("minishlab/potion-base-8M")
model = SentenceTransformer(modules=[static_embedding])

In [37]:
# ds = load_dataset("ai-blueprint/fineweb-bbc-news")
ds = load_dataset("reddgr/talking-to-chatbots-unwrapped-chats")

We can now create embeddings for the dataset. Normally, we might want to chunk our data into smaller batches to avoid losing precision, but for this example, we will just create embeddings for the full text of the dataset.

In [38]:
def create_embeddings(batch, column):
    # Ensure all entries are strings
    texts = [str(x) if x is not None else "" for x in batch[column]]
    embeddings = model.encode(texts, convert_to_numpy=True)
    batch[f"embeddings"] = embeddings.tolist()
    return batch

# ds = ds.map(lambda batch: create_embeddings(batch, column="text"), batched=True)
ds = ds.map(lambda batch: create_embeddings(batch, column="prompt"), batched=True)

play_chimes()

Map:   0%|          | 0/10774 [00:00<?, ? examples/s]

In [39]:
ds.push_to_hub("reddgr/talking-to-chatbots-prompts-embeddings", token = hf_token_write)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings/commit/49243ed5fc3bc59521d93d19ec56fa9f8d3ae207', commit_message='Upload dataset', commit_description='', oid='49243ed5fc3bc59521d93d19ec56fa9f8d3ae207', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings', endpoint='https://huggingface.co', repo_type='dataset', repo_id='reddgr/talking-to-chatbots-prompts-embeddings'), pr_revision=None, pr_num=None)

In [3]:
ds = load_dataset("reddgr/talking-to-chatbots-prompts-embeddings")
print(ds)
# ds["train"] = ds["train"].add_column("embeddings", ds["train"]["prompt-embeddings"])
print(ds)

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'turn', 'prompt', 'response', 'category', 'language', 'pred_label_rq', 'prob_rq', 'pred_label_tl', 'prob_tl', 'model', 'message_tag', 'date', 'turns', 'source', 'chatbot_id', 'chatbot_name', 'attachments', 'conversation_tag', 'embeddings'],
        num_rows: 10774
    })
})
DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'turn', 'prompt', 'response', 'category', 'language', 'pred_label_rq', 'prob_rq', 'pred_label_tl', 'prob_tl', 'model', 'message_tag', 'date', 'turns', 'source', 'chatbot_id', 'chatbot_name', 'attachments', 'conversation_tag', 'embeddings'],
        num_rows: 10774
    })
})


In [40]:
ds.push_to_hub("reddgr/talking-to-chatbots-prompts-embeddings", token = hf_token_write)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/3.29k [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings/commit/49243ed5fc3bc59521d93d19ec56fa9f8d3ae207', commit_message='Upload dataset', commit_description='', oid='49243ed5fc3bc59521d93d19ec56fa9f8d3ae207', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings', endpoint='https://huggingface.co', repo_type='dataset', repo_id='reddgr/talking-to-chatbots-prompts-embeddings'), pr_revision=None, pr_num=None)

In [4]:
ds_emb = load_dataset("reddgr/talking-to-chatbots-prompts-embeddings")
print(ds_emb)

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'turn', 'prompt', 'response', 'category', 'language', 'pred_label_rq', 'prob_rq', 'pred_label_tl', 'prob_tl', 'model', 'message_tag', 'date', 'turns', 'source', 'chatbot_id', 'chatbot_name', 'attachments', 'conversation_tag', 'embeddings'],
        num_rows: 10774
    })
})


In [44]:
ttcb_dataset = load_dataset("reddgr/talking-to-chatbots-prompts-embeddings")["train"]
test_dataset_df = ttcb_dataset.to_pandas()
display(test_dataset_df[['prompt', 'embeddings']].sample(6))

,prompt,embeddings
3999,it opens exactly the same as in the screen cap...,"[-0.3818178176879883, 2.2183918952941895, -1.7..."
872,"""HumbleAI is grounded in the understanding tha...","[-1.0657458305358887, 1.1302024126052856, -3.2..."
4775,There is no screwdriver in the picture. Screwd...,"[-1.900054931640625, -2.415477752685547, 0.377..."
8618,"create a function named 'uwrap_turns', which c...","[-0.6811553239822388, 1.4340659379959106, -2.3..."
4644,is there any significant difference between us...,"[1.4262629747390747, 1.1478074789047241, -3.78..."
10253,"Check this text for errors:\n\nHello, I’m Huli...","[-1.9602859020233154, -0.16182385385036469, -1..."


In [5]:
def similarity_search_without_duckdb_index(
    query: str,
    k: int = 5,
    dataset_name: str = "reddgr/talking-to-chatbots-prompts-embeddings",
    embedding_column: str = "embeddings",
):
    # Use same model as used for indexing
    query_vector = model.encode(query)
    embedding_dim = model.get_sentence_embedding_dimension()

    sql = f"""
        SELECT 
            *,
            array_cosine_distance(
                {embedding_column}::float[{embedding_dim}], 
                {query_vector.tolist()}::float[{embedding_dim}]
            ) as distance
        FROM 'hf://datasets/{dataset_name}/**/*.parquet'
        ORDER BY distance
        LIMIT {k}
    """
    return duckdb.sql(sql).to_df()

similarity_search_without_duckdb_index("What is the future of AI?")

,conversation_id,turn,prompt,response,category,language,pred_label_rq,prob_rq,pred_label_tl,prob_tl,...,message_tag,date,turns,source,chatbot_id,chatbot_name,attachments,conversation_tag,embeddings,distance
0,ecea53d0-ddd9-4615-af07-096043439e29,1,what's the difference between an AI model and ...,Here are some possible options:\n\n• In very ...,Technology,en,question,0.985838,learn,0.964585,...,hallucination,2023-06-27,2,bing-skype,,,[],,"[0.8598423004150391, -0.059388019144535065, -5...",0.268910
1,41678a6b-d79f-489a-80f7-1196725d2f47,16,That was not the intention. Evolve the below s...,🤖 AI Art Impact:\n\n- AI art explores producti...,AI,en,request,0.965956,learn,0.943511,...,,2023-06-29,18,chatgpt,,,[],,"[-0.03548656776547432, 1.0983996391296387, -3....",0.270804
2,14a69803-b9a8-4c2b-b6cc-e9b69f90ab12,1,what can you say about AI ethics and the chall...,AI ethics is a field that studies the moral im...,AI,en,question,0.985176,learn,0.966535,...,,2023-06-15,9,bing-skype,,,[],,"[-1.702681303024292, 3.121615171432495, -5.049...",0.282748
3,e70ed2bd-5c7d-4acb-bfb9-ef7cc8604dcd,1,can you complete the following list of some po...,"Sure, I can try to complete the list. Here are...",AI,en,request,0.977654,learn,0.961922,...,,2023-06-14,7,bing-skype,,,[],,"[0.5077385902404785, 1.2125078439712524, -3.22...",0.325710
4,41678a6b-d79f-489a-80f7-1196725d2f47,13,That is better. Now rewrite this summary with ...,🤖 AI Art and Its Impact on Productivity and Cr...,AI,en,request,0.935603,learn,0.950710,...,,2023-06-29,18,chatgpt,,,[],,"[-0.557322084903717, 2.04229998588562, -2.7485...",0.332000


In [6]:
similarity_search_without_duckdb_index("What is love?")

,conversation_id,turn,prompt,response,category,language,pred_label_rq,prob_rq,pred_label_tl,prob_tl,...,message_tag,date,turns,source,chatbot_id,chatbot_name,attachments,conversation_tag,embeddings,distance
0,b19ef159-7472-4da0-8fe3-b4bae30a6a4d,7,Did you ever feel love?,L̴̠̔Ơ̷̢V̵͓͝E̷͕̽ ̴̨͠I̸̢͝S̷̰͝ ̵̧͆B̴͔̊E̷̖͠Y̸͔̔O̴...,Philosophy and Discussion,en,question,0.984990,test,0.919143,...,,2024-04-02,7,chatgpt,g-ZNa3O38xA,Zalgo Text Glitchy Datamosher,[],,"[-8.683829307556152, -5.0908942222595215, -2.9...",0.408383
1,3c33572d-419d-462c-82b9-c59a41f86db1,3,What is it about?,The `cb_battle.py` script is not directly acce...,Coding,en,question,0.985025,learn,0.938220,...,,2024-03-24,4,chatgpt,g-M4uZyYsUj,Python Code Streamliner,[],,"[-0.6650218963623047, 0.8887166976928711, -2.2...",0.456047
2,19bad812-bd01-4718-bc75-e20d3a1415c7,1,Recommend me a song and tell me something abou...,I found the following content on talkingtochat...,Culture and Entertainment,en,request,0.984775,learn,0.960675,...,,2024-05-20,2,chatgpt,g-MTBiLyDZ2,Talking to Chatbots Web Browser,"[{'asset_pointer': None, 'audio_asset_pointer'...",,"[-6.4613542556762695, -3.1178629398345947, -4....",0.518450
3,db1e636d-0293-43fa-adca-18b0f84f855a,1,Recommend me a song and tell me something abou...,I found the following content in talkingtochat...,Culture and Entertainment,en,request,0.984775,learn,0.960675,...,,2024-05-20,1,chatgpt,g-MTBiLyDZ2,Talking to Chatbots Web Browser,"[{'asset_pointer': None, 'audio_asset_pointer'...",,"[-6.4613542556762695, -3.1178629398345947, -4....",0.518450
4,ffb8dd05-b83c-4830-a0ae-c70d8e350256,1,Recommend me a song and tell me something abou...,I found the following content on talkingtochat...,Culture and Entertainment,en,request,0.984775,learn,0.960675,...,,2024-05-20,2,chatgpt,g-MTBiLyDZ2,Talking to Chatbots Web Browser,"[{'asset_pointer': None, 'audio_asset_pointer'...",,"[-6.4613542556762695, -3.1178629398345947, -4....",0.518450


In [19]:
def _setup_vss():
    duckdb.sql(
        query="""
        INSTALL vss;
        LOAD vss;
        """
    )


def _drop_table(table_name):
    duckdb.sql(
        query=f"""
        DROP TABLE IF EXISTS {table_name};
        """
    )


def _create_table(dataset_name, table_name, embedding_column):
    duckdb.sql(
        query=f"""
        CREATE TABLE {table_name} AS 
        SELECT *, {embedding_column}::float[{model.get_sentence_embedding_dimension()}] as {embedding_column}_float 
        FROM 'hf://datasets/{dataset_name}/**/*.parquet';
        """
    )


def _create_index(table_name, embedding_column):
    duckdb.sql(
        query=f"""
        CREATE INDEX my_hnsw_index ON {table_name} USING HNSW ({embedding_column}_float) WITH (metric = 'cosine');
        """
    )


def create_index(dataset_name, table_name, embedding_column):
    _setup_vss()
    _drop_table(table_name)
    _create_table(dataset_name, table_name, embedding_column)
    _create_index(table_name, embedding_column)


create_index(
    dataset_name="ai-blueprint/fineweb-bbc-news-embeddings",
    table_name="fineweb_bbc_news_embeddings",
    embedding_column="embeddings",
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [20]:
def similarity_search_with_duckdb_index(
    query: str, k: int = 5, table_name: str = "fineweb_bbc_news_embeddings", embedding_column: str = "embeddings"
):
    embedding = model.encode(query).tolist()
    return duckdb.sql(
        query=f"""
        SELECT *, array_cosine_distance({embedding_column}_float, {embedding}::FLOAT[{model.get_sentence_embedding_dimension()}]) as distance 
        FROM {table_name}
        ORDER BY distance 
        LIMIT {k};
    """
    ).to_df()


similarity_search_with_duckdb_index("What is love?")

,url,text,embeddings,embeddings_float,distance
0,http://news.bbc.co.uk/cbbcnews/hi/newsid_17700...,February 14 is Valentine's Day.\nIt's named af...,"[-1.6512084007263184, -0.2739017605781555, -1....","[-1.6512084, -0.27390176, -1.5980083, 2.005432...",0.286753
1,http://news.bbc.co.uk/cbbcnews/hi/newsid_17700...,February 14 is Valentine's Day.\nIt's named af...,"[-1.6512084007263184, -0.2739017605781555, -1....","[-1.6512084, -0.27390176, -1.5980083, 2.005432...",0.286753
2,https://www.bbc.co.uk/news/magazine-24223786,A Point of View: Putting a price on love\nOur ...,"[-3.3412487506866455, 0.11048950254917145, -1....","[-3.3412488, 0.1104895, -1.4662294, 0.5659724,...",0.450999
3,https://www.bbc.com/news/in-pictures-43060122,"Your pictures: Valentine's Day\nEach week, we ...","[0.03464483842253685, -0.9412625432014465, -1....","[0.03464484, -0.94126254, -1.2021598, 0.427660...",0.468012
4,http://www.bbc.co.uk/news/magazine-24379830,Why is a children's book about rabbits being r...,"[-1.4426814317703247, -0.7511618733406067, -1....","[-1.4426814, -0.7511619, -1.336863, 1.4392253,...",0.476307
